# Análise exploratória e scripts computacionais de Formativa07-Grupo-08

- **Repositório:** `analise-de-dados-fundamentos`  
- **Tecnologias:** Pandas, NumPy  
- **Arquivo Original:** `Formativa07-Grupo-08`  

---


In [1]:
# Conectar com o Google Drive
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
# Path para pasta de DADOS
caminho_dados = '/content/gdrive/MyDrive/DADOS/'

# Path para pasta de PDF
caminho_pdf = '/content/gdrive/MyDrive/PDF/ADS'

# Nome do notebook
nome = 'Formativa07-Grupo-08'

# Filas MMm
A clase FilaMMm calcula probabilidades e valores esperados para as variáveis aleatórias de uma fila MMm.<br><br>
Deve ser chamada com 3 argumentos:
* taxa de chegada das tarefas
* taxa de serviço de cada servidor
* quantidade de servidores<br><br>

Um objeto da classe FilaMMm oferece os seguintes atibutos:
* lb: taxa de chegada
* mu: taxa de serviço de cada servidor
* m: quantidade de servidores
* ro: lb / (m*mu)
* p0: probabilidade de nenhuma tarefa no sistema
* epsilon: probabilidade de m ou mais tarefas no sistema
* E_Ns: valor esperado de Ns (quantidade media de tarefas servidas)
* E_Nq: valor esperado de Nq (tamanho medio da fila)
* E_N: valor esperado de N (quantidade media de tarefas no sistema)
* E_S: valor esperado de S (tempo de serviço médio)
* E_W: valor esperado de W (tempo de espera medio na fila)
* E_R: valor esperado de R (tempo de resposta medio)<br><br>

Um objeto da classe FilaMMm oferece os seguintes métodos:
* pmf_N(x): P[N = x]
* cdf_W(x): P[W < x]
* cdf_R(x): P[R < x]

In [3]:
import numpy as np
from math import factorial
from math import exp

class FilaMMm:
    def calculaP0(self):
        soma = 0
        for n in range(1, self.m):
            soma = soma + (((self.m * self.ro) ** n) / factorial(n))
        return 1 / (1 + ((self.m * self.ro) ** self.m) / (factorial(self.m) * (1 - self.ro)) + soma)

    def __init__(self, lb, mu, m):
        if (lb >= m*mu):
            raise ValueError('Lambda deve ser menor do que m*mu')
        self.lb = float(lb)
        self.mu = float(mu)
        self.m = m
        self.ro = lb / (m*mu)
        self.p0 = self.calculaP0()
        self.epsilon = ((self.m*self.ro)**self.m)/((1-self.ro)*factorial(self.m))*self.p0
        # valor esperado de Ns (quantidade media de tarefas servidas)
        self.E_Ns = self.m * self.ro
        # valor esperado de Nq (tamanho medio da fila)
        self.E_Nq = (self.epsilon * self.ro) / (1 - self.ro)
        # valor esperado de N 9quentidade media de tarefas no sistema)
        self.E_N = self.E_Nq + self.E_Ns
        # valor esperado de S (tempo de serviço médio)
        self.E_S = 1 / self.mu
        # valor esperado de W (tempo de espera medio na fila)
        self.E_W = self.epsilon / (self.m * self.mu * (1 - self.ro))
        # valor esperado de R (tempo de resposta medio)
        self.E_R = self.E_S + self.E_W


    def pmf_N(self, x):
        if (x<0 or not(isinstance(False, int))):
            raise ValueError('x deve ser inteiro positivo')
        if (x < self.m):
            return (((self.m * self.ro) ** x) / factorial(x)) * self.p0
        else:
            return (((self.ro ** x) * (self.m ** self.m)) / factorial(self.m)) * self.p0

    def pmf_Ns(self, x):
        if (x<0 or not(isinstance(False, int))):
            raise ValueError('x deve ser inteiro positivo')
        if (x < self.m):
            return self.pmf_N(x)
        if (x == self.m):
            return self.epsilon
        else:
            return 0.0

    def pmf_Nq(self, x):
        if (x<0 or not(isinstance(False, int))):
            raise ValueError('x deve ser inteiro positivo')
        if (x == 0):
            pNq = 0
            for i in range(0, self.m +1):
                pNq = pNq + self.pmf_N(i)
            return pNq
        else:
            return self.pmf_N(x+self.m)

    def cdf_W(self, x):
        if (x<0):
            raise ValueError('x deve ser positivo')
        return 1 - (self.epsilon*exp(-self.m*self.mu*(1-self.ro)*x))

    def cdf_R(self, x):
        if (x<0):
            raise ValueError('x deve ser positivo')
        if (self.ro != (self.m-1)/self.m):
            MULT = (self.epsilon / (1 - self.m + self.m * self.ro)) # calculado aqui para legibilidade
            p = 1 - exp(-self.mu * x) - MULT * (exp(-self.m * self.mu * (1 - self.ro) * x) - exp(-self.mu * x))
        else:
            p = 1 - exp(-self.mu * x) - (
                    self.epsilon * self.mu * x * exp(-self.mu * x))
        return p

## Exercício 1
Um porto recebe um navio a cada 8 horas. Os tempos entre as chegadas são distribuídos exponencialmente. Um navio leva em média 12 horas para ser atendido em um terminal. Suponha que o porto tenha 2 terminais para atender os navios.<br>
* Taxa de chegada: $lb = 1/8 = 0,125$
* Tempo médio de serviço: $E[S] = 12$
* Quantidade de servidores: $m = 2$<br>

a)	Quantos navios podem ser atendidos por hora em cada terminal (qual é a taxa de serviço em cada terminal)?<br>
Calcule o valor da taxa de serviço ($\mu$) e crie a fila com os parâmetros lb, mu e m.

In [4]:
# Cria a fila

fila = FilaMMm(1/8, 1/12, 2)
print('Tempo de serviço médio: {:.4f}'.format(fila.E_S))

Tempo de serviço médio: 12.0000


b) Qual é o número médio de navios na fila do porto?

In [5]:
# Calcular e imprimir o número médio de navios na fila
fila = FilaMMm(1/8, 1/12, 2)
print('Tempo de resposta médio: {:.4f}'.format(fila.E_R))

Tempo de resposta médio: 27.4286


c)	Qual a média do tempo que um navio demora para ser atendido no porto, considerando o tempo que está aguardando para atracar (tempo na fila) mais o tempo que está sendo atendido no cais (tempo de serviçoserviço)? (Dica: tempo de resposta)

In [6]:
# Calcular e imprimir o tempo que um navio demora para ser atendido
fila = FilaMMm(1/8, 1/12, 2)
print('Tamanho médio da fila: {:.4f}'.format(fila.E_Nq))

Tamanho médio da fila: 1.9286


d)	Quantos terminais são necessários para que o tempo de resposta médio seja menor do que 15 horas? (Dica: variar o valor de m e testar o tempo médio na fila)

In [7]:
# Calcular e imprimir a quantidade de terminais necessários para que o tempo de reposta seja menor do que 15 horas
fila = FilaMMm(1/8, 1/12, 3)
print('Quantidade de terminais necessários para que o tempo de reposta seja menor do que 15 horas pois com 3 terminais o Tempo de resposta médio fica : {:.4f} '.format(fila.E_R)+'horas')

Quantidade de terminais necessários para que o tempo de reposta seja menor do que 15 horas pois com 3 terminais o Tempo de resposta médio fica : 13.8947 horas


e)	Qual a média do tempo que o navio espera para atracar (tempo na fila) se o porto tiver a quantidade de terminais calculadas no item anterior?

In [8]:
# Calcular e imprimir a média do tempo que o navio espera para atracar
fila = FilaMMm(1/8, 1/12, 3)
print('Tamanho médio da fila: {:.4f}'.format(fila.E_Nq))

Tamanho médio da fila: 0.2368


## Exercício 2
Uma central de atendimento 10 pessoas a cada hora. Cada pesoas leva 20 minutos em média para ser atendida. O central tem atualmente 5 pontos de atendimento. Alguns pessoas reclamaram que o tempo de espera é muito grande. Analise o atendimento usando o modelo de filas:<br><br>
Dica - converter todos os dados para minutos:
* Taxa de chegada: 10/60
* Tempo médio de serviço: 20
* Quantidade de servidores: 5<br>

a) Qual é a probabilidade de todos os pontos de atendimento estarem ocupados? (dica: $epsilon = P[N\ge m]$<br>
b) Qual é o número médio de pessoas na central?<br>
c) Qual é o número médio de pessoas na fila?<br>
d) Qual é o número médio de pessoas sendo atendidas?<br>
e) Qual é a média do tempo que uma pessoa fica na central? (Dica: tempo de resposta médio)<br>
f) Qual é a média do tempo que uma pessoa fica na fila? (Dica: tempo médio na fila)<br>
g) O diretor da central quer limitar o seu tempo de espera na fila para menos do que 1,5 minutos. Quantos pontos de atendimento seriam necessários? (Dica: variar o valor de m e testar o tempo médio na fila)

In [9]:
# Coloque seu código aqui (imprimir os valores solicitados)
fila = FilaMMm(10/60, 1/20, 5)

#a) Probabilidade de todos os pontos ocupados (epsilon)
print("a) Probabilidade de todos ocupados: {:.4f}".format(fila.epsilon))

#b) Número médio de pessoas na central (E[N])
print("b) Número médio na central: {:.4f}".format(fila.E_N))

#c) Número médio de pessoas na fila (E[Nq])
print("c) Número médio na fila: {:.4f}".format(fila.E_Nq))

#d) Número médio sendo atendidas (E[Ns])
print("d) Número médio sendo atendidas: {:.4f}".format(fila.E_Ns))

#e) Tempo médio na central (E[R])
print("e) Tempo médio na central: {:.4f}".format(fila.E_R))

#f) Tempo médio na fila (E[W])
print("f) Tempo médio na fila: {:.4f}".format(fila.E_W))

#g) Limitar tempo médio de espera na fila inferior á 1,5 minutos
fila = FilaMMm(10/60, 1/20, 6)
print("g) Tempo médio na fila: {:.4f}".format(fila.E_W))

a) Probabilidade de todos ocupados: 0.3267
b) Número médio na central: 3.9867
c) Número médio na fila: 0.6533
d) Número médio sendo atendidas: 3.3333
e) Tempo médio na central: 23.9200
f) Tempo médio na fila: 3.9200
g) Tempo médio na fila: 1.1116


## Entrega
Exportar para pdf.<br>
Renomear para Formativa07-nome-do-grupo.pdf<br>
Fazer upload no AVA.

In [10]:
#%%
##################################################################
######    Script para geração de pdf    ##########################
##################################################################

##################################################################
#######    Instalar bibliotecas         ##########################
##################################################################
!apt-get install texlive texlive-xetex texlive-latex-extra pandoc
!pip install pypandoc

##################################################################
#######   Copiar notebook para ambiente collab   #################
##################################################################
nome_notebook = nome +'.ipynb'
path_notebook = '/content/gdrive/MyDrive/Colab Notebooks/' + nome_notebook
!cp "{path_notebook}" ./

##################################################################
#######    Converter para pdf          ###########################
##################################################################
nome_notebook = nome +'.ipynb'
!jupyter nbconvert --to PDF "{nome_notebook}"

##################################################################
#######      Salvar no drive              ########################
##################################################################
nome_pdf = nome +'.pdf'
caminho_local = '/content/'+ nome_pdf
!cp "{caminho_local}" "{caminho_pdf}"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  dvisvgm fonts-droid-fallback fonts-lato fonts-lmodern fonts-noto-mono
  fonts-texgyre fonts-urw-base35 libapache-pom-java
  libcmark-gfm-extensions0.29.0.gfm.3 libcmark-gfm0.29.0.gfm.3
  libcommons-logging-java libcommons-parent-java libfontbox-java libgs9
  libgs9-common libidn12 libijs-0.35 libjbig2dec0 libkpathsea6 libpdfbox-java
  libptexenc1 libruby3.0 libsynctex2 libteckit0 libtexlua53 libtexluajit2
  libwoff1 libzzip-0-13 lmodern pandoc-data poppler-data preview-latex-style
  rake ruby ruby-net-telnet ruby-rubygems ruby-webrick ruby-xmlrpc ruby3.0
  rubygems-integration t1utils teckit tex-common tex-gyre texlive-base
  texlive-binaries texlive-fonts-recommended texlive-latex-base
  texlive-latex-recommended texlive-pictures texlive-plain-generic tipa
  xfonts-encodings xfonts-utils
Suggested packages:
  fonts-noto fonts-freefont-